# Path B - Gold vs CPI and US Dollar Index (Continues from PathA)

PathA (gold only) is preserved at Visual-Gold-Explorer-PathA.ipynb. This file reuses its loaders and adds macro overlays. If you’ve seen PathA, skip to Piece 8.

Dataset is collected from `Dataset/gold-price.csv`, `Dataset/CPIAUCSL.csv`, and `Dataset/US Dollar Index Historical Data.csv`.

## Piece 1 — Loading the Helpers and Libraries
Install helpers once, then we see the data, just like unpacking puzzle pieces.

In [2]:
# Run this once. If it says 'already satisfied', we're good to go.
# We use plotly = interactive lines, ipywidgets = sliders
%pip install -q pandas plotly ipywidgets openpyxl
print("Setup done — charts are now zoomable Wand draggable")

Note: you may need to restart the kernel to use updated packages.
Setup done — charts are now zoomable Wand draggable


In [3]:
import numpy as np
# Compatibility patch: some libraries still call np.matrix which was removed in NumPy 2.0
# This keeps the notebook working on both NumPy 1.x and 2.x without downgrading
if not hasattr(np, 'matrix'):
    np.matrix = np.asmatrix  # type: ignore[attr-defined]
    print("patched np.matrix for NumPy 2.x compatibility")

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from ipywidgets import interact, IntSlider, IntRangeSlider
import pathlib

# Library check
print("Libraries ready:", pd.__version__)

Libraries ready: 3.0.3


## Piece 2 — Load and Feel the Data (Gold Price First)
We load the `gold-price.csv`. `US Dollar Index Historical Data.csv`, and `CPIAUCSL.csv` before opening the monthly album of `gold-price.csv`.

In [4]:
# List the 3 datasets we use for Path B -- gold only (PathA) + CPI + Dollar
# This loop reads them all so we don't repeat pd.read_csv 3 times
import pathlib as _pl

dataset_list = [
    ("gold", "gold-price.csv", ["date"], "date"),
    ("cpi", "CPIAUCSL.csv", ["observation_date"], "observation_date"),
    ("dxy", "US Dollar Index Historical Data.csv", ["Date"], "Date"),
]

# Resolve base: try relative Dataset/ first, fallback to absolute
base_rel = _pl.Path("Dataset")
base_abs = _pl.Path(r"D:\Daniel\Data Science Project\Visual-Gold-Explorer-Why-Gold-Moves-in-Crisis-with-Honest-Baseline-\Dataset")

dfs = {}
for name, fname, parse_cols, sort_col in dataset_list:
    csv_path = base_rel / fname
    if not csv_path.exists():
        csv_path = base_abs / fname
    # Read with parse_dates where possible; DXY uses MM/DD/YYYY which pandas handles, but we coerce if needed
    df = pd.read_csv(csv_path, parse_dates=parse_cols)
    # Fallback for DXY if Date stayed as string
    if name == "dxy" and df[sort_col].dtype == object:
        df[sort_col] = pd.to_datetime(df[sort_col], format="%m/%d/%Y", errors="coerce")
    df = df.sort_values(sort_col)
    dfs[name] = df
    print(f"{name}: {len(df)} rows | {sort_col} {df[sort_col].min()} -> {df[sort_col].max()} | file: {csv_path.name}")

# Unpack for later cells (keeps old variable names so rest of notebook still works)
gold = dfs["gold"]


print(f"\nRows: {len(gold)} | From {gold['date'].min().date()} to {gold['date'].max().date()}")
print(gold.head(3).to_string(index=False))
print("---")
print(gold.tail(3).to_string(index=False))
print("\nScroll the output. 1960 = $35 flat (fixed price), 2026 = ~$4000. That's the story.")

gold: 799 rows | date 1960-01-01 00:00:00 -> 2026-07-01 00:00:00 | file: gold-price.csv
cpi: 955 rows | observation_date 1947-01-01 00:00:00 -> 2026-07-01 00:00:00 | file: CPIAUCSL.csv
dxy: 12056 rows | Date 1979-12-26 00:00:00 -> 2026-09-02 00:00:00 | file: US Dollar Index Historical Data.csv

Rows: 799 | From 1960-01-01 to 2026-07-01
      date  gold_price_usd
1960-01-01              35
1960-02-01              35
1960-03-01              35
---
      date  gold_price_usd
2026-05-01            4587
2026-06-01            4228
2026-07-01            4073

Scroll the output. 1960 = $35 flat (fixed price), 2026 = ~$4000. That's the story.


In [5]:
# Missing value check for the gold price dataset
print("Missing values:", gold["gold_price_usd"].isna().sum())
print("Example check — Sep 2000:")
print(gold[gold["date"]=="2000-09-01"])

Missing values: 0
Example check — Sep 2000:
          date  gold_price_usd
488 2000-09-01             274


## Piece 3 — Quick Recap from PathA

This section recaps the shape of the gold price data, the crises bands, and the same moving average logic as PathA.

In [6]:
crises = [
    ("2007-12-01", "2009-06-01", "2008 Financial Crisis"),
    ("2020-02-01", "2020-04-01", "Covid Crash"),
    ("2022-01-01", "2022-12-01", "2022 Inflation Shock"),
]

print(f"Recap: gold {len(gold)} months (1960-2026), crises {len(crises)} bands, same MA logic as PathA")

Recap: gold 799 months (1960-2026), crises 3 bands, same MA logic as PathA


## Piece 4 — Two Sliders: Year Range & Moving Average

Sliders are movable to trigger the line reaction inside the chart.

---
**Slider 1: Year range** (like cropping a photo)

**Slider 2: Moving average (MA)** (like smoothing wrinkles. MA=3 = wiggly, MA=12 = smooth yearly trend (since data is monthly, 12 = 1-year average))

In [ ]:
# Prepare a helper column: moving average
# For monthly data, window=12 means 12-month average
def plot_with_sliders(year_start=2000, year_end=2026, ma_window=12, show=True):
    # plot_with_sliders' 3 params are failsafe defaults when interact is not present
    # show=True -> display in Jupyter (for Piece 5), show=False -> silent return for HTML export (for final cell)
    subset = gold[(gold["date"].dt.year >= year_start) & (gold["date"].dt.year <= year_end)].copy()
    subset["MA"] = subset["gold_price_usd"].rolling(window=ma_window, min_periods=1).mean()
    
    goldfig = go.Figure()
    goldfig.add_trace(go.Scatter(x=subset["date"], y=subset["gold_price_usd"], mode="lines", name="Monthly Price",
                                 line=dict(width=1.5, color="gray"), opacity=0.7))
    goldfig.add_trace(go.Scatter(x=subset["date"], y=subset["MA"], mode="lines", name=f"{ma_window}-Month Avg",
                                 line=dict(width=3, color="#d4a017")))
    # Keep crisis shading only if in range
    for start, end, label in crises:
        if str(year_start) <= start <= str(year_end) or str(year_start) <= end <= str(year_end):
            goldfig.add_vrect(x0=start, x1=end, fillcolor="gray", opacity=0.15, line_width=0)
    goldfig.update_layout(title=f"Gold {year_start}-{year_end} | MA={ma_window} months -- Move Sliders Below",
                          xaxis_title="Year", yaxis_title="USD/oz", height=420)
    if show:
        goldfig.show()
        return None
    else:
        return goldfig

print("Sliders ready. Move them below -- watch the gold line smooth or crop.")


Sliders ready. Move them below -- watch the gold line smooth or crop.


In [8]:
# Three sliders for the year range and the moving average window
from ipywidgets import fixed
interact(plot_with_sliders,
         year_start=IntSlider(value=2000, min=1960, max=2024, step=1, description="Start Year"),
         year_end=IntSlider(value=2026, min=1961, max=2026, step=1, description="End Year"),
         ma_window=IntSlider(value=12, min=1, max=36, step=1, description="MA Months"),
         show=fixed(True));

interactive(children=(IntSlider(value=2000, description='Start Year', max=2024, min=1960), IntSlider(value=202…

## Piece 5 — Load and Feel the Consumer Price Index and USD Index
We open the monthly album of `US Dollar Index Historical Data.csv` and `CPIAUCSL.csv`.

In [9]:
cpi = dfs["cpi"]
dxy = dfs["dxy"]

print(f"\nCPI Rows: {len(cpi)} | From {cpi['observation_date'].min().date()} to {cpi['observation_date'].max().date()}")

print(cpi.head(3).to_string(index=False))
print("---")
print(cpi.tail(3).to_string(index=False))

print("===========================")

print(f"DXY Rows: {len(dxy)} | From {dxy['Date'].min().date()} to {dxy['Date'].max().date()}")

print(dxy.head(3).to_string(index=False))
print("---")
print(dxy.tail(3).to_string(index=False))


CPI Rows: 955 | From 1947-01-01 to 2026-07-01
observation_date  CPIAUCSL
      1947-01-01     21.48
      1947-02-01     21.62
      1947-03-01     22.00
---
observation_date  CPIAUCSL
      2026-05-01   333.979
      2026-06-01   332.568
      2026-07-01   332.813
DXY Rows: 12056 | From 1979-12-26 to 2026-09-02
      Date  Price  Open  High   Low  Vol. Change %
1979-12-26  85.99 85.99 85.99 85.99   NaN   -0.21%
1979-12-27  85.51 85.51 85.51 85.51   NaN   -0.56%
1979-12-28  85.81 85.81 85.81 85.81   NaN    0.35%
---
      Date  Price  Open  High   Low  Vol. Change %
2026-08-31  99.43 99.70 99.71 99.39   NaN   -0.27%
2026-09-01  99.68 99.42 99.72 99.35   NaN    0.25%
2026-09-02  99.76 99.77 99.81 99.67   NaN    0.09%


In [10]:
# Missing values check for CPI and DXY datasets
print(f"CPI Missing Values: {cpi['CPIAUCSL'].isna().sum()}")
print(f"DXY Missing Values: {dxy['Price'].isna().sum()}")

CPI Missing Values: 1
DXY Missing Values: 0


In [11]:
# Handle missing values
cpi["CPIAUCSL"] = cpi["CPIAUCSL"].interpolate(method="linear")
print(f"CPI missing after interpolation: {cpi['CPIAUCSL'].isna().sum()}")

dxy = dxy[["Date", "Price"]].copy()
dxy["Price"] = pd.to_numeric(dxy["Price"], errors="coerce")

CPI missing after interpolation: 0


## Piece 6 — Merge Gold Price, Consumer Price Index, and USD Index Datasets
- We resample USD Index (DXY) to monthly frequency
- Merge Gold Price, Consumer Price Index, and USD Index Datasets
- Verify the oldest/latest date, and rows/missing values count
- **(Optional)** Save the merged datasets to a CSV file for last step verification

In [12]:
# Resample DXY to monthly frequency, taking the last available value for each month
dxy_monthly = dxy.set_index('Date').resample('MS').last()

# Merge the datasets on their respective date columns
merged = gold.merge(cpi, left_on="date", right_on="observation_date", how="inner").merge(dxy_monthly, left_on="date", right_on="Date", how="inner")

# Check the merged dataset
print(f"Merged: {len(merged)} months | {merged['date'].min().date()} -> {merged['date'].max().date()}")
print(merged.head(3).to_string(index=False))
print("------------------------------------------------------------")
print(merged.tail(3).to_string(index=False))
print("------------------------------------------------------------")
print(f"Rows count: {merged.count()}")
print("-------------------------------------------")
print(f"Missing values count: {merged.isna().sum()}")

# Save the merged dataset to a CSV file for last step verification and further analysis
# merged_to_csv = merged.to_csv(
#    "testing_data.csv",
#    index=False
# )

Merged: 560 months | 1979-12-01 -> 2026-07-01
      date  gold_price_usd observation_date  CPIAUCSL  Price
1979-12-01             455       1979-12-01      76.9  85.82
1980-01-01             675       1980-01-01      78.0  86.14
1980-02-01             665       1980-02-01      79.0  87.65
------------------------------------------------------------
      date  gold_price_usd observation_date  CPIAUCSL  Price
2026-05-01            4587       2026-05-01   333.979  98.91
2026-06-01            4228       2026-06-01   332.568 101.19
2026-07-01            4073       2026-07-01   332.813  99.91
------------------------------------------------------------
Rows count: date                560
gold_price_usd      560
observation_date    560
CPIAUCSL            560
Price               560
dtype: int64
-------------------------------------------
Missing values count: date                0
gold_price_usd      0
observation_date    0
CPIAUCSL            0
Price               0
dtype: int64


## Piece n — Export the Chart to HTML (Later)
Exporting the chart to HTML to make the charts interactable.

In [14]:
# Export static charts to ONE HTML -- for GitHub (shows even without Jupyter)
# fig, fig2, fig4 are fully zoomable in HTML. Slider figure is snapshot-only (ipywidgets don't work in HTML).
# For reviewers: use GIF/screenshots of the slider in action (see README) -- that's the GIF + static snapshots approach.
# import plotly.io as pio

# Create silent snapshot of the slider at default (no display in this cell)
# fig_slider_snapshot = plot_with_sliders(year_start=2000, year_end=2026, ma_window=12)

# with open("Visual-Gold-Explorer-PathA-interactive.html", "w", encoding="utf-8") as f:
#     f.write(fig.to_html(full_html=True, include_plotlyjs='cdn'))
#     f.write("<h2 style='font-family:sans-serif; text-align:center; margin-top:40px;'>Gold with Crisis Shading</h2>")
#     f.write(fig2.to_html(full_html=False, include_plotlyjs=False))
#     f.write("<h2 style='font-family:sans-serif; text-align:center; margin-top:40px;'>Interactive Slider Snapshot (for HTML -- sliders work only in Jupyter)</h2>")
#     f.write("<p style='font-family:sans-serif; text-align:center; color:#555;'>In Jupyter, drag the sliders in Piece 5. On GitHub, this is a static snapshot at 2000-2026 / MA 12. See README GIF for live drag.</p>")
#     f.write(fig_slider_snapshot.to_html(full_html=False, include_plotlyjs=False))
#     f.write("<img src='assets/slider-drag.gif' alt='Slider Drag GIF' style='display:block; margin-left:auto; margin-right:auto; width:60%;'>")
#     f.write("<h2 style='font-family:sans-serif; text-align:center; margin-top:40px;'>Honest Baseline: Actual vs Naive Forecasts</h2>")
#     f.write(fig4.to_html(full_html=False, include_plotlyjs=False))
# print("Saved 4 charts to Visual-Gold-Explorer-PathA-interactive.html -- add this ONE file to GitHub")